# 04 - Analise Estatistica Inicial da Segmentacao Bruta

Materializa a base analitica da segmentacao bruta a partir do SQLite do projeto, persiste os resultados iniciais da analise estatistica e exibe um resumo por modelo.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import (
    MetricsCollector,
    build_and_persist_analysis_segmentacao_bruta_resumo_modelo,
)
from src.repositories import (
    AnaliseSegmentacaoBrutaBaseRepository,
    AnaliseSegmentacaoBrutaResumoModeloRepository,
)


## Carregamento da base analitica

Le a avaliacao persistida no SQLite, monta a base analitica por `imagem + modelo + execucao` e grava a tabela `analise_segmentacao_bruta_base`.


In [ ]:
collector = MetricsCollector(force_recalculate=False)
base_repository = AnaliseSegmentacaoBrutaBaseRepository()

df_base = collector.persist_analysis_segmentacao_bruta_base(
    base_repository=base_repository,
)

print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes: {df_base["execucao"].nunique()}')

df_base.head()


## Resumo estatistico inicial por modelo

Calcula as estatisticas descritivas iniciais das metricas brutas e grava a tabela `analise_segmentacao_bruta_resumo_modelo`.


In [ ]:
resumo_repository = AnaliseSegmentacaoBrutaResumoModeloRepository()
df_resumo_modelo, _ = build_and_persist_analysis_segmentacao_bruta_resumo_modelo(
    df_base=df_base,
    repository=resumo_repository,
)

print(f'Registros no resumo por modelo: {len(df_resumo_modelo)}')
df_resumo_modelo


## Leitura orientada do resultado

Mostra uma visao compacta das medias e medianas por modelo e por metrica para orientar a etapa seguinte do notebook.


In [ ]:
pd.pivot_table(
    df_resumo_modelo,
    index='modelo',
    columns='metric_name',
    values=['mean', 'median'],
)
